# 🟥 Familia 3 · Transformer-based Models for Raw Audio
**Detección de Parkinson en voz — TFM**

---
Modelos de esta familia:
- **`temporal_transformer`** — Transformer encoder con patches de waveform
- **`cnn_transformer_hybrid`** — Front-end CNN + Transformer encoder

**Flujo:** Edita la celda `⚙️ CONFIGURACION` → `Kernel → Restart & Run All`

## ⚙️ CONFIGURACION — Edita aquí antes de ejecutar

In [ ]:
MODEL_NAME   = "cnn_transformer_hybrid"  # Opciones: "temporal_transformer" | "cnn_transformer_hybrid"
DATASET_NAME = "neurovoz"                 # Opciones: "neurovoz" | "pc-gita"
FOLD_INDEX   = 0                          # Fold: 0..4

MODEL_HYPERPARAMS = {
    # === Para temporal_transformer ===
    "patch_size":       64,    # Muestras por patch (más pequeño → más tokens, más coste)

    # === Compartidos ===
    "d_model":          128,   # Dimensión del embedding del Transformer
    "nhead":            4,     # Número de cabezas de atención (debe dividir d_model)
    "num_layers":       4,     # Capas del encoder Transformer
    "dim_feedforward":  256,   # Dimensión de la capa FFN interna
    "dropout":          0.1,

    # === Para cnn_transformer_hybrid ===
    "cnn_channels": [32, 64, 128],  # Front-end CNN: canales por bloque
}

TRAIN_CONFIG = {
    "epochs":       60,
    "batch_size":   16,   # Reducir si OOM por ser modelos más grandes
    "lr":           5e-4,
    "weight_decay": 1e-4,
    "patience":     12,
    "augment":      True,
}

NOTES = "Prueba inicial cnn_transformer_hybrid fold 0"

print(f"Modelo: {MODEL_NAME} | Dataset: {DATASET_NAME} | Fold: {FOLD_INDEX + 1}")

## 1. Imports

In [ ]:
import sys, os
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import torch, torch.nn as nn, json, datetime
import matplotlib.pyplot as plt

from src.models.transformer_audio import TRANSFORMER_MODELS
from src.training.audio_dataset import get_dataloaders, get_class_weights
from src.training.trainer import Trainer
from src.training.experiment_logger import ExperimentLogger
from src.evaluation.metrics import calculate_metrics

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {DEVICE}")

## 2. Datos

In [ ]:
SPLITS_JSON = os.path.join(PROJECT_ROOT, "data", "data_splits.json")
DATA_ROOT   = os.path.join(PROJECT_ROOT, "data", "processed")

train_loader, val_loader, test_loader = get_dataloaders(
    splits_json=SPLITS_JSON, dataset_name=DATASET_NAME, fold_index=FOLD_INDEX,
    data_root=DATA_ROOT, batch_size=TRAIN_CONFIG["batch_size"], augment_train=TRAIN_CONFIG["augment"]
)

with open(SPLITS_JSON) as f:
    splits = json.load(f)
class_weights = get_class_weights(splits[DATASET_NAME][FOLD_INDEX]["train_files"]).to(DEVICE)
print(f"Pesos de clase: {class_weights.tolist()}")

## 3. Modelo

In [ ]:
ModelClass = TRANSFORMER_MODELS[MODEL_NAME]

if MODEL_NAME == "temporal_transformer":
    model = ModelClass(
        patch_size      = MODEL_HYPERPARAMS["patch_size"],
        d_model         = MODEL_HYPERPARAMS["d_model"],
        nhead           = MODEL_HYPERPARAMS["nhead"],
        num_layers      = MODEL_HYPERPARAMS["num_layers"],
        dim_feedforward = MODEL_HYPERPARAMS["dim_feedforward"],
        dropout         = MODEL_HYPERPARAMS["dropout"],
        num_classes     = 2,
    )
elif MODEL_NAME == "cnn_transformer_hybrid":
    model = ModelClass(
        cnn_channels    = MODEL_HYPERPARAMS["cnn_channels"],
        d_model         = MODEL_HYPERPARAMS["d_model"],
        nhead           = MODEL_HYPERPARAMS["nhead"],
        num_layers      = MODEL_HYPERPARAMS["num_layers"],
        dim_feedforward = MODEL_HYPERPARAMS["dim_feedforward"],
        dropout         = MODEL_HYPERPARAMS["dropout"],
        num_classes     = 2,
    )

model = model.to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"{MODEL_NAME} | Parámetros: {n_params:,}")

## 4. Entrenamiento

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=TRAIN_CONFIG["lr"], weight_decay=TRAIN_CONFIG["weight_decay"])
criterion = nn.CrossEntropyLoss(weight=class_weights)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=TRAIN_CONFIG["epochs"], eta_min=1e-6
)
trainer = Trainer(model, optimizer, criterion, DEVICE)

history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
best_val_loss, patience_counter, best_state = float("inf"), 0, None

for epoch in range(TRAIN_CONFIG["epochs"]):
    tr_loss, tr_preds, tr_labels = trainer.train_epoch(train_loader)
    vl_loss, vl_preds, vl_probs, vl_labels = trainer.evaluate(val_loader)

    tr_acc = sum(p == l for p, l in zip(tr_preds, tr_labels)) / len(tr_labels)
    vl_acc = sum(p == l for p, l in zip(vl_preds, vl_labels)) / len(vl_labels)
    history["train_loss"].append(tr_loss); history["val_loss"].append(vl_loss)
    history["train_acc"].append(tr_acc);   history["val_acc"].append(vl_acc)
    scheduler.step()

    print(f"Epoch {epoch+1:03d} | Train {tr_loss:.4f}/{tr_acc:.3f} | Val {vl_loss:.4f}/{vl_acc:.3f} | LR {scheduler.get_last_lr()[0]:.2e}")

    if vl_loss < best_val_loss:
        best_val_loss = vl_loss; patience_counter = 0
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    else:
        patience_counter += 1
        if patience_counter >= TRAIN_CONFIG["patience"]:
            print(f"Early stopping en epoch {epoch+1}"); break

model.load_state_dict(best_state)
print(f"\nMejor Val Loss: {best_val_loss:.4f}")

## 5. Curvas

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ep = range(1, len(history["train_loss"]) + 1)
axes[0].plot(ep, history["train_loss"], label="Train", color="steelblue")
axes[0].plot(ep, history["val_loss"],   label="Val",   color="tomato")
axes[0].set_title("Loss"); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(ep, history["train_acc"], label="Train", color="steelblue")
axes[1].plot(ep, history["val_acc"],   label="Val",   color="tomato")
axes[1].set_title("Accuracy"); axes[1].set_ylim([0,1]); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.suptitle(f"{MODEL_NAME.upper()} — {DATASET_NAME} Fold {FOLD_INDEX+1}", fontsize=14)
plt.tight_layout(); plt.show()

## 6. Evaluación en Test

In [ ]:
_, test_preds, test_probs, test_labels = trainer.evaluate(test_loader)
metrics = calculate_metrics(test_labels, test_preds, test_probs)
print("\n" + "="*50)
print(f" METRICAS TEST SET ({DATASET_NAME} — Fold {FOLD_INDEX+1})")
print("="*50)
for k, v in metrics.items():
    print(f"  {k:<20}: {v:.4f}")

## 7. Guardar y registrar

In [ ]:
MODELS_SAVE_DIR = os.path.join(PROJECT_ROOT, "experiments", "saved_models")
os.makedirs(MODELS_SAVE_DIR, exist_ok=True)
ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
model_path = os.path.join(MODELS_SAVE_DIR, f"{ts}_{MODEL_NAME}_{DATASET_NAME}_fold{FOLD_INDEX+1}.pt")
torch.save({"model_name": MODEL_NAME, "hyperparams": MODEL_HYPERPARAMS,
            "state_dict": model.state_dict(), "metrics": metrics,
            "dataset": DATASET_NAME, "fold": FOLD_INDEX + 1}, model_path)
print(f"Modelo guardado: {model_path}")

logger = ExperimentLogger(log_dir=os.path.join(PROJECT_ROOT, "experiments"))
logger.log(MODEL_NAME, DATASET_NAME, FOLD_INDEX+1, {**MODEL_HYPERPARAMS, **TRAIN_CONFIG}, metrics, history, NOTES)
logger.print_summary()